# EE 391 — Lecture 10 In-Class Assignment
## Independence and Correlation: Titanic Dataset

**Work in pairs or groups of 3.  Fill in every cell marked `# YOUR CODE HERE`.  You should use the internet to look stuff up (code documentation, concepts), but please do not simply feed this notebook into generative AI for a solution**

This notebook has four parts:

| Part | Topic | Connects to |
|------|-------|-------------|
| 1 | Joint PMF and marginals | Lecture 9 |
| 2 | Testing for independence | Lecture 9–10 |
| 3 | Covariance and correlation — by hand and via pandas | Lecture 10 |
| 4 | Visualising relationships | Lecture 10 |

---

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# Load the dataset
df = pd.read_csv("./data/titanic.csv")

print('Shape:', df.shape)
df.head()

Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


**Variable descriptions**

| Column | Type | Description |
|--------|------|-------------|
| `survived` | 0 / 1 | Did the passenger survive? |
| `pclass` | 1, 2, 3 | Ticket class (1 = first, 3 = third) |
| `sex` | male / female | Sex |
| `age` | float | Age in years (~20% missing) |
| `sibsp` | int | Number of siblings/spouses aboard |
| `parch` | int | Number of parents/children aboard |
| `fare` | float | Ticket fare (£) |
| `embarked` | S / C / Q | Port of embarkation |

---
## Part 1 — Joint PMF and Marginals

### 1a.  Compute the joint PMF of `survived` and `pclass`

Using `pd.crosstab` with `normalize=True`, construct the **empirical joint PMF**
$p_{S,C}(s, c) = P(\text{survived}=s,\ \text{pclass}=c)$.

- Rows should be `survived` ∈ {0, 1}
- Columns should be `pclass` ∈ {1, 2, 3}
- Every cell should be a probability (between 0 and 1, all cells sum to 1)

In [ ]:
# YOUR CODE HERE
joint = ...

print('Joint PMF p(survived, pclass):')
print(joint)
print('\nSum of all cells (should be 1.0):', joint.values.sum().round(6))

### 1b.  Compute the marginal PMFs

From the joint PMF, compute:
- $p_S(s) = P(\text{survived} = s)$ — by summing across columns
- $p_C(c) = P(\text{pclass} = c)$ — by summing across rows

*Hint: `joint.sum(axis=...)` — which axis collapses which variable?*

In [ ]:
# YOUR CODE HERE
p_survived = ...
p_pclass   = ...

print('Marginal p(survived):')
print(p_survived)
print('\nMarginal p(pclass):')
print(p_pclass)

### 1c.  Conditional PMF:  P(survived | pclass)

Compute the conditional PMF $p_{S|C}(s\,|\,c)$ for each class.
This is the survival *rate* within each class.

- Divide each column of the joint PMF by the corresponding marginal probability of that class
- Check: each column of your result should sum to 1

In [ ]:
# YOUR CODE HERE
p_surv_given_class = ...

print('P(survived | pclass):')
print(p_surv_given_class.round(3))
print('\nColumn sums (each should be 1.0):')
print(p_surv_given_class.sum(axis=0).round(6))

**Q1.  Looking at P(survived=1 | pclass), how does survival probability vary across the three classes?  Does this match your prior expectation?**

*Your answer:*

---
## Part 2 — Testing for Independence

Recall: `survived` and `pclass` are **independent** if and only if
$$p_{S,C}(s, c) = p_S(s) \cdot p_C(c) \quad \text{for ALL pairs } (s, c)$$

### 2a.  Construct the independence table

Build the table of values $p_S(s) \cdot p_C(c)$ — the joint PMF *if* the two variables were independent.

*Hint: `np.outer(p_survived.values, p_pclass.values)` — wrap in a DataFrame with the right index and columns.*

In [ ]:
# YOUR CODE HERE
independent_table = ...

print('Independence table p(s) * p(c):')
print(independent_table.round(4))

### 2b.  Compare: joint vs independence table

Compute the cell-by-cell difference: `joint - independent_table`.
If the variables were independent, every cell would be zero.

In [ ]:
# YOUR CODE HERE
diff = ...

print('Difference (joint − independence table):')
print(diff.round(4))

**Q2.  Are `survived` and `pclass` independent?  Cite specific cells from the difference table to support your answer.  What is the sign of the difference for pclass=1?  What does the sign tell you?**

*Your answer:*

### 2c.  Visualise the comparison as a grouped bar chart

Plot the **empirical joint PMF** and the **independence table** side-by-side for each (survived, pclass) cell.
This makes the deviation from independence visible at a glance.

Use `matplotlib`.  Suggested structure:
- x-axis: pclass (1, 2, 3)
- two groups of bars per class: `survived=0` and `survived=1`
- two colours per bar: actual (solid) vs independence (hatched or lighter)

*One approach: loop over survived values and use `ax.bar` with an offset for each group.*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

# YOUR CODE HERE

ax.set_xlabel('Passenger class')
ax.set_ylabel('Probability')
ax.set_title('Joint PMF: actual vs independence assumption')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 3 — Covariance and Correlation

We now compute $\text{cov}(X, Y)$ and $\rho(X, Y)$ for pairs of **numeric** variables, first by hand and then using pandas.

### 3a.  Compute cov(fare, pclass) by hand

Use the formula
$$\text{cov}(X, Y) = E[XY] - E[X]\,E[Y]$$

Treat the dataset as the population (use the empirical mean as $E[X]$).  Do not use `.cov()` yet.

In [ ]:
X = df['fare'].values
Y = df['pclass'].values

# YOUR CODE HERE
E_X  = ...
E_Y  = ...
E_XY = ...
cov_hand = ...

print(f'E[fare]         = {E_X:.4f}')
print(f'E[pclass]       = {E_Y:.4f}')
print(f'E[fare * pclass]= {E_XY:.4f}')
print(f'cov(fare,pclass)= {cov_hand:.4f}')

### 3b.  Verify using numpy

`np.cov(X, Y)` returns a 2×2 covariance **matrix**.  The off-diagonal entry is the covariance.  What would it make sense for the diagonal to be?

Note: by default numpy uses $n-1$ (sample covariance).  Pass `ddof=0` to use $n$ (population covariance) and match your answer above.

In [ ]:
# YOUR CODE HERE
cov_matrix = ...
cov_numpy  = ...

print(f'numpy cov(fare, pclass) [ddof=0]: {cov_numpy:.4f}')
print(f'Matches hand calculation: {np.isclose(cov_hand, cov_numpy)}')

**Q3.  The covariance is negative.  Interpret this in plain language: what does it say about how fare and passenger class are related?**

*Your answer:*

### 3c.  Compute ρ(fare, pclass) by hand

$$\rho(X, Y) = \frac{\text{cov}(X, Y)}{\sqrt{\text{Var}(X) \cdot \text{Var}(Y)}}$$

In [ ]:
# YOUR CODE HERE
var_X  = ...
var_Y  = ...
rho_hand = ...

print(f'Var(fare)   = {var_X:.4f}')
print(f'Var(pclass) = {var_Y:.4f}')
print(f'rho(fare, pclass) = {rho_hand:.4f}')

### 3d.  Full numeric correlation matrix via pandas

Compute the correlation matrix for all numeric columns using `df.corr()`.
Check that your ρ(fare, pclass) from 3c appears in the correct cell.

In [ ]:
# YOUR CODE HERE — select numeric columns and call .corr()
numeric_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr_matrix  = ...

print(corr_matrix.round(3))

**Q4.  Look at the `survived` row (or column) of the correlation matrix.**
- **Which variable has the strongest (most negative) correlation with survival?**
- **Which variable has the weakest correlation with survival?**
- **Does a near-zero ρ(age, survived) mean age had no effect on who survived?  Why or why not?  (Think back to Slide 5 from today.)**

*Your answer:*

---
## Part 4 — Visualising Relationships

Three different plot types — each suited to a different kind of variable pair.

### 4a.  Correlation heatmap  (continuous × continuous)

Use `sns.heatmap` to visualise the correlation matrix from 3d.

Requirements:
- Diverging colormap centred at 0 (e.g. `cmap='coolwarm'`)
- Annotate each cell with its ρ value
- Set `vmin=-1, vmax=1` so the scale is always the full ρ range

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

# YOUR CODE HERE

ax.set_title('Pearson correlation matrix — Titanic numeric variables')
plt.tight_layout()
plt.show()

### 4b.  Grouped box plot  (continuous × categorical)

Box plots are good for showing the distribution of a continuous variable across categories.
Plot the distribution of **`fare`** for each **`pclass`**.

Use `sns.boxplot(data=df, x='pclass', y='fare')`.
Then add a horizontal line at the overall mean fare.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# YOUR CODE HERE

ax.set_title('Fare distribution by passenger class')
ax.set_xlabel('Passenger class')
ax.set_ylabel('Fare (£)')
plt.tight_layout()
plt.show()

**Q5.  The box plot shows the spread within each class, not just the mean.  Is the variance of fare similar across classes, or does it differ?  How does this relate to the covariance you computed in Part 3?**

*Your answer:*

### 4c.  Stacked bar chart of survival rates  (categorical × categorical)

For each combination of **`pclass`** and **`sex`**, plot the survival rate as a stacked or grouped bar chart.

Suggested approach:
1. Compute a pivot table: rows = pclass, columns = sex, values = mean(survived)
2. Call `.plot(kind='bar')` on the pivot table

This is a visual version of the conditional PMF P(survived=1 | pclass, sex).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# YOUR CODE HERE
# Step 1: pivot table
survival_by_class_sex = ...

# Step 2: plot

ax.set_title('Survival rate by class and sex')
ax.set_xlabel('Passenger class')
ax.set_ylabel('P(survived = 1)')
ax.set_ylim(0, 1)
ax.axhline(df.survived.mean(), color='black', linestyle='--', linewidth=1, label='Overall mean')
ax.legend()
plt.tight_layout()
plt.show()

### 4d.  Scatter plot with a third variable encoded in colour  (two continuous + one categorical)

Plot **`age`** (x-axis) vs **`fare`** (y-axis), with each point coloured by **`survived`**.
Drop rows where age is missing first.

Use `ax.scatter` with the `c` argument for colour, or `sns.scatterplot` with `hue='survived'`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

df_clean = df.dropna(subset=['age'])

# YOUR CODE HERE

ax.set_xlabel('Age (years)')
ax.set_ylabel('Fare (£)')
ax.set_title('Age vs Fare, coloured by survival')
plt.tight_layout()
plt.show()

**Q6.  The scatter plot shows age vs fare coloured by survival.  ρ(age, survived) is close to zero — does the scatter plot suggest age truly had no relationship with survival, or is something more subtle going on?  What kind of relationship (if any) does the colour pattern suggest?**

*Your answer:*

---
## Stretch goal — if your group finishes early

**Var(X + Y) in practice.**

Let $X$ = `fare` and $Y$ = `age` (drop missing ages).

1. Compute $\text{Var}(X + Y)$ directly from the data.
2. Compute $\text{Var}(X) + \text{Var}(Y) + 2\,\text{cov}(X, Y)$ using the formula from lecture.
3. Verify they match.
4. What fraction of $\text{Var}(X+Y)$ comes from the covariance term?  Is the covariance term large or negligible here?

In [ ]:
df_s = df.dropna(subset=['age']).copy()

# YOUR CODE HERE
